# Indian Jail Management System – AIML Project
**CSL236 | Units 1, 2, 3 and 5 | Unit 4 (Neural Networks) excluded**

> **Important:** The companion Excel workbook contains synthetic demonstration data, not actual Indian prison, prisoner, officer, legal or live-capacity records. Facility names and officer names are fictional. The offence reference is illustrative only. This notebook is for coursework and must not be used to make bail, detention, guilt, sentencing or prison-placement decisions.

## Execution order
Run from top to bottom. Every model has a separate sequence of cells: **model definition → training → prediction → evaluation → confusion matrix/graph**. Regression models have actual-vs-predicted and residual plots. Clustering has its own evaluation and visualizations.

**Revised version:** Original lesson/model sequence retained; fixes include safe reference-label merging, optional Apriori fallback, sample-size validation, robust quantile bins and cross-validation checks. Place the `.xlsx` beside this notebook (or in `/mnt/data`).

**Dataset compatibility update:** This notebook reads `Indian_Jail_Management_Complete_Synthetic_Dataset.xlsx` and maps its `_Demo` fields onto the original workflow. Classification labels are illustrative fixed reference labels; perfect holdout metrics should not be interpreted as genuine legal generalization.

**New edition:** Companion workbook `Indian_Jail_Management_Synthetic_Expanded_Validated.xlsx`. Classification labels are synthetic operational review priority, not legal bailability.

## Unit 1 – Introduction to AI and Machine Learning

- **AI:** broad field of building systems that perform tasks associated with intelligent behaviour.
- **ML:** methods that learn patterns from data.
- **Supervised learning:** labelled data; includes classification and regression.
- **Unsupervised learning:** unlabelled data; includes clustering and dimensionality reduction.
- **Reinforcement learning:** an agent learns through rewards and penalties; not required for this project.
- **Deep learning:** ML using multi-layer neural networks; Unit 4 is excluded as instructed.
- **Structured data:** tables; **semi-structured:** JSON-like records; **unstructured:** documents, images and audio.
- **AI vs ML vs deep learning:** AI is the broadest area; ML is one approach within AI; deep learning is a subset of ML.

**Brief history and trends:** symbolic AI and expert systems preceded statistical ML; larger datasets and computing accelerated modern ML and deep learning. Current trends include explainable AI, responsible AI, generative AI, privacy-preserving analytics and human-in-the-loop systems.

**Project applications:** aggregate population forecasting, occupancy monitoring, data-quality checks and administrative resource planning. No algorithm should decide an individual's legal status.

## 0. Import libraries and load the workbook

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score, silhouette_score)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, SVR
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
# Apriori is optional in some Jupyter/Colab environments. If absent, use a
# compact, compatible fallback for the small teaching basket in Section 5.7.
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    print("Apriori backend: mlxtend")
except ImportError:
    from itertools import combinations
    def apriori(df, min_support=0.5, use_colnames=True):
        if not use_colnames:
            raise ValueError("Fallback expects use_colnames=True")
        names=list(df.columns)
        supports={}
        for r in range(1,len(names)+1):
            candidates=[]
            if r==1:
                candidates=[frozenset([x]) for x in names]
            else:
                prev={k for k in supports if len(k)==r-1}
                candidates=[frozenset(xs) for xs in combinations(names,r)
                            if all(frozenset(s) in prev for s in combinations(xs,r-1))]
            if not candidates: break
            found=0
            for itemset in candidates:
                support=float(df[list(itemset)].all(axis=1).mean())
                if support>=min_support:
                    supports[itemset]=support
                    found+=1
            if not found: break
        return pd.DataFrame([{"support":v,"itemsets":k} for k,v in supports.items()],
                            columns=["support","itemsets"])
    def association_rules(df, metric="confidence", min_threshold=0.5, **kwargs):
        supports={row.itemsets:row.support for row in df.itertuples(index=False)}
        rows=[]
        for items, joint in supports.items():
            if len(items)<2: continue
            for r in range(1,len(items)):
                for left in combinations(items,r):
                    ant=frozenset(left); con=items-ant
                    if ant not in supports or con not in supports: continue
                    sa,sc=supports[ant],supports[con]
                    confidence=joint/sa
                    lift=confidence/sc
                    if {"confidence":confidence,"lift":lift,"support":joint}.get(metric,confidence)<min_threshold: continue
                    rows.append({"antecedents":ant,"consequents":con,"antecedent support":sa,
                                 "consequent support":sc,"support":joint,"confidence":confidence,
                                 "lift":lift,"leverage":joint-sa*sc,
                                 "conviction":(1-sc)/(1-confidence) if confidence<1 else float('inf')})
        return pd.DataFrame(rows,columns=["antecedents","consequents","antecedent support",
            "consequent support","support","confidence","lift","leverage","conviction"])
    print("Apriori backend: built-in fallback (install mlxtend for full library functionality)")

# New workbook adapter: preserve the original project analysis and column interfaces.
FILE = "Indian_Jail_Management_Synthetic_Expanded_Validated.xlsx"
if not os.path.exists(FILE):
    for candidate in ["/mnt/data/" + FILE, "./data/" + FILE]:
        if os.path.exists(candidate):
            FILE = candidate
            break
if not os.path.exists(FILE):
    try:
        from google.colab import files
        uploaded = files.upload()
        candidates = [name for name in uploaded if name.startswith(
            "Indian_Jail_Management_Synthetic_Expanded_Validated") and name.endswith(".xlsx")]
        if not candidates:
            raise FileNotFoundError("Upload Indian_Jail_Management_Synthetic_Expanded_Validated.xlsx.")
        FILE = candidates[0]
    except ImportError:
        raise FileNotFoundError("Place Indian_Jail_Management_Synthetic_Expanded_Validated.xlsx beside this notebook.")

xls = pd.ExcelFile(FILE)
print("Workbook:", FILE)
print("Available worksheets:", xls.sheet_names)
required_sheets = ["Facility_Master_Demo", "Prison_Monthly_Demo",
                   "Classification_Cases_Demo", "Legal_Offence_Reference",
                   "Regression_Training_Demo"]
missing_sheets = sorted(set(required_sheets) - set(xls.sheet_names))
if missing_sheets:
    raise ValueError(f"Missing required sheets: {missing_sheets}")
facility = pd.read_excel(xls, "Facility_Master_Demo")
monthly_raw = pd.read_excel(xls, "Prison_Monthly_Demo")
cases_raw = pd.read_excel(xls, "Classification_Cases_Demo")
offence_ref = pd.read_excel(xls, "Legal_Offence_Reference")
reg_raw = pd.read_excel(xls, "Regression_Training_Demo")

# Map newer demonstrative fields to names used throughout the original notebook.
facility = facility.rename(columns={
    "Facility_Name_Demo": "Facility_Name", "Facility_Type_Demo": "Facility_Type",
    "Authorized_Capacity_Demo": "Capacity"})
monthly_raw = monthly_raw.rename(columns={
    "Facility_Name_Demo": "Facility_Name", "Facility_Type_Demo": "Facility_Type",
    "Authorized_Capacity_Demo": "Capacity", "Inmate_Population_Demo": "Population",
    "Admissions_Demo": "Admissions", "Releases_Demo": "Releases",
    "Transfers_In_Demo": "Transfers_In", "Transfers_Out_Demo": "Transfers_Out",
    "Occupancy_Rate_pct_Demo": "Occupancy_Pct"})
cases_raw = cases_raw.rename(columns={
    "Case_ID_Demo": "Case_ID", "Case_Stage_Demo": "Case_Stage",
    "Case_Review_Priority_Synthetic": "Case_Review_Priority_Synthetic"})
offence_ref = offence_ref.rename(columns={
    "Legal_Bailability": "Legal_Bailability"})
reg_raw = reg_raw.rename(columns={
    "Authorized_Capacity": "Capacity",
    "Target_Population_Next_Month": "Target_Next_Month_Population"})

print("Facility", facility.shape, "Monthly", monthly_raw.shape,
      "Classification", cases_raw.shape, "Regression", reg_raw.shape)
print("Synthetic demonstration data: model metrics do not validate real-world legal decisions.")


## 1. Inspect data – head, tail, info and descriptive statistics

In [ ]:
display(monthly_raw.head())
display(cases_raw.head())
display(reg_raw.head())
display(facility.head())

In [ ]:
print("Monthly data info"); monthly_raw.info()
print("\\nClassification data info"); cases_raw.info()

In [ ]:
display(monthly_raw.describe(include="all").T)
display(cases_raw.describe(include="all").T)

# Unit 2 – Data Preprocessing and Feature Engineering
All preprocessing steps are shown explicitly. Modelling pipelines fit imputers, encoders and scalers on training data only to prevent leakage.

## 2.1 Missing-value analysis

In [ ]:
def missing_report(df):
    return pd.DataFrame({"Missing":df.isna().sum(),
                         "Percent":(df.isna().mean()*100).round(2)}).sort_values("Percent",ascending=False)
display(missing_report(monthly_raw))
display(missing_report(cases_raw))
display(missing_report(reg_raw))

## 2.2 Duplicate detection and removal

In [ ]:
print("Duplicate rows – monthly:",monthly_raw.duplicated().sum())
print("Duplicate rows – classification:",cases_raw.duplicated().sum())
monthly = monthly_raw.drop_duplicates().copy()
cases = cases_raw.drop_duplicates().copy()
reg = reg_raw.drop_duplicates().copy()

## 2.3 Data types and date conversion

In [ ]:
monthly["Month"] = pd.to_datetime(monthly["Month"],errors="coerce")
reg["Feature_Month"] = pd.to_datetime(reg["Feature_Month"],errors="coerce")
for col in ["Capacity","Population","Admissions","Releases","Transfers_In","Transfers_Out","Occupancy_Pct"]:
    monthly[col] = pd.to_numeric(monthly[col],errors="coerce")
display(monthly.dtypes)

## 2.4 Check invalid values and logical consistency

In [ ]:
print("Non-positive capacity:",(monthly.Capacity<=0).sum())
print("Negative population:",(monthly.Population<0).sum())
print("Negative admissions:",(monthly.Admissions<0).sum())
monthly = monthly[(monthly.Capacity>0)&(monthly.Population>=0)].copy()
monthly["Occupancy_Recalculated"] = monthly.Population/monthly.Capacity*100
display(monthly[["Capacity","Population","Occupancy_Pct","Occupancy_Recalculated"]].head())

## 2.5 Missing-value treatment – numerical median and categorical mode

In [ ]:
monthly_clean = monthly.copy()
for col in monthly_clean.select_dtypes(include=np.number):
    monthly_clean[col] = monthly_clean[col].fillna(monthly_clean[col].median())
for col in monthly_clean.select_dtypes(exclude=np.number):
    if monthly_clean[col].isna().any():
        monthly_clean[col] = monthly_clean[col].fillna(monthly_clean[col].mode().iloc[0])
print("Missing values after demonstration treatment:",monthly_clean.isna().sum().sum())

## 2.6 EDA – population distribution

In [ ]:
plt.figure(figsize=(8,4)); plt.hist(monthly_clean.Population,bins=35)
plt.title("Distribution of synthetic monthly population"); plt.xlabel("Population"); plt.ylabel("Frequency"); plt.show()

## 2.7 EDA – occupancy distribution and capacity threshold

In [ ]:
plt.figure(figsize=(8,4)); plt.hist(monthly_clean.Occupancy_Recalculated,bins=35)
plt.axvline(100,ls="--",label="100% of capacity"); plt.legend()
plt.title("Occupancy distribution"); plt.xlabel("Occupancy (%)"); plt.ylabel("Frequency"); plt.show()
print("Rows above 100% occupancy:",(monthly_clean.Occupancy_Recalculated>100).sum())

## 2.8 EDA – monthly population trend and moving-average smoothing

In [ ]:
trend=monthly_clean.groupby("Month").Population.sum().sort_index()
smooth=trend.rolling(3,min_periods=1).mean()
plt.figure(figsize=(12,4)); plt.plot(trend,label="Monthly total"); plt.plot(smooth,label="3-month moving average")
plt.title("Monthly population trend – synthetic data"); plt.legend(); plt.ylabel("Population"); plt.show()

## 2.9 EDA – state-wise average occupancy

In [ ]:
state_occ=monthly_clean.groupby("State_UT").Occupancy_Recalculated.mean().sort_values(ascending=False)
display(state_occ.head(15).to_frame("Average occupancy (%)"))
plt.figure(figsize=(12,5)); state_occ.plot(kind="bar"); plt.ylabel("Average occupancy (%)")
plt.title("State/UT average synthetic occupancy"); plt.tight_layout(); plt.show()

## 2.10 EDA – correlation matrix

In [ ]:
corr=monthly_clean.select_dtypes(include=np.number).corr()
plt.figure(figsize=(9,7)); plt.imshow(corr,aspect="auto"); plt.colorbar()
plt.xticks(range(len(corr)),corr.columns,rotation=90); plt.yticks(range(len(corr)),corr.columns)
plt.title("Correlation matrix"); plt.tight_layout(); plt.show()

## 2.11 Outlier detection using IQR (detect only; do not blindly delete genuine overcrowding)

In [ ]:
for col in ["Population","Admissions","Releases","Occupancy_Recalculated"]:
    q1,q3=monthly_clean[col].quantile([.25,.75]); iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    print(col,"IQR bounds:",round(lo,2),round(hi,2),
          "| outlier count:",int(((monthly_clean[col]<lo)|(monthly_clean[col]>hi)).sum()))

## 2.12 Standardization and normalization

In [ ]:
scale_cols=["Capacity","Population","Admissions","Releases"]
sample=monthly_clean[scale_cols].copy()
standardized=pd.DataFrame(StandardScaler().fit_transform(sample),columns=scale_cols)
normalized=pd.DataFrame(MinMaxScaler().fit_transform(sample),columns=scale_cols)
display(standardized.head()); display(normalized.head())

## 2.13 Categorical encoding – one-hot and label encoding demonstration

In [ ]:
from sklearn.preprocessing import LabelEncoder
display(pd.get_dummies(cases[["Act","Cognizable_Status"]].head(8),columns=["Act","Cognizable_Status"]))
le=LabelEncoder()
label_demo=cases[["Cognizable_Status"]].head(8).copy()
label_demo["Encoded"] = le.fit_transform(label_demo["Cognizable_Status"])
display(label_demo)  # Codes are category identifiers, not meaningful ranks.

## 2.14 Feature engineering – lagged population and next-month target

In [ ]:
reg["Feature_Month"]=pd.to_datetime(reg.Feature_Month)
display(reg[["Feature_Month","Population_Current","Population_Lag_1M",
             "Population_Lag_3M","Population_Lag_6M","Target_Next_Month_Population"]].head(10))

## 2.15 Dimensionality reduction – PCA

In [ ]:
pca_cols=["Capacity","Population","Admissions","Releases","Transfers_In","Transfers_Out"]
Xp=StandardScaler().fit_transform(monthly_clean[pca_cols])
pca=PCA(n_components=2,random_state=42); pca_xy=pca.fit_transform(Xp)
print("Explained variance:",pca.explained_variance_ratio_,
      "Total:",pca.explained_variance_ratio_.sum())
plt.figure(figsize=(7,4)); plt.scatter(pca_xy[:,0],pca_xy[:,1],alpha=.2)
plt.title("PCA – monthly records"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()

# Unit 3 – Supervised Classification: Synthetic Administrative Case-Review Priority

**Target:** `Case_Review_Priority_Synthetic` (High vs Standard). This is a fictional workflow-triage label; **it is not legal bailability, guilt, sentence length, detention or release advice**. The target is generated probabilistically from case administrative fields, with noise. Offence identifiers are not used to predict the target. Do not use this demonstration to make real decisions about people.

## 3.1 Classification preparation – prevent lookup leakage and split first

Use real-valued administrative features and categorical case attributes, excluding legal labels, case IDs, Act/Section, and outcome-adjacent fields. Train/test preprocessing occurs inside each pipeline.

In [ ]:
# Classification target is a stochastic, CASE-LEVEL synthetic administrative label.
# Do not merge labels from Legal_Offence_Reference: that would reintroduce lookup leakage.
cls=cases.copy()
target='Case_Review_Priority_Synthetic'
num_cls=['Number_of_Charges_Demo','Number_of_Accused_Demo',
         'Prior_Convictions_Count_Demo','Pending_Days_Demo','Hearings_Held_Demo']
cat_cls=['Cognizable_Status','Case_Stage','Legal_Aid_Status_Demo','Documentation_Status_Demo']
features=num_cls+cat_cls
assert target not in features
assert all(v not in features for v in ['Act','Section','Offence_Description','Case_ID'])
X=cls[features].copy()
for col in num_cls:
    X[col]=pd.to_numeric(X[col],errors='coerce')
y=cls[target].astype(str)
counts=y.value_counts()
assert counts.min()>=2, 'Each class must have at least two rows'
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
def classification_preprocessor():
    return ColumnTransformer([
      ('num',Pipeline([('impute',SimpleImputer(strategy='median')),
                       ('scale',MinMaxScaler())]),num_cls),
      ('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                       ('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cls)])
print('Train/test:', X_train.shape,X_test.shape)
print('Train balance:',y_train.value_counts(normalize=True).round(3).to_dict())
from sklearn.dummy import DummyClassifier
baseline=DummyClassifier(strategy='most_frequent').fit(X_train,y_train)
print('Majority-class baseline accuracy:',round(accuracy_score(y_test,baseline.predict(X_test)),3))


## 3.2 Classification evaluation helper
Accuracy alone is not enough. We report precision, recall, F1 and confusion matrix. Weighted scores account for class frequencies.

In [ ]:
classification_results=[]
def classification_evaluation(name,model):
    pred=model.predict(X_test)
    acc=accuracy_score(y_test,pred)
    pre=precision_score(y_test,pred,average="weighted",zero_division=0)
    rec=recall_score(y_test,pred,average="weighted",zero_division=0)
    f1=f1_score(y_test,pred,average="weighted",zero_division=0)
    classification_results.append([name,acc,pre,rec,f1])
    print(name)
    print(classification_report(y_test,pred,zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test,pred, cmap="Blues")
    plt.title(name+" – Confusion Matrix"); plt.show()
    print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")
    return pred

## 3.3 Model 1 – Logistic Regression\n**Step 1: define and train the model.**

In [ ]:
logistic_model = Pipeline([
    ("preprocess",classification_preprocessor()),
    ("model",LogisticRegression(max_iter=1500, class_weight='balanced'))
])
logistic_model.fit(X_train,y_train)
print("Logistic Regression training completed.")

### Model 1 – Prediction

In [ ]:
logistic_pred = logistic_model.predict(X_test)
print("First 15 predictions:",logistic_pred[:15])
print("First 15 actual labels:",y_test.iloc[:15].to_list())

### Model 1 – Evaluation and confusion matrix

In [ ]:
# Reuse the prediction already generated above for consistent evaluation.
acc=accuracy_score(y_test,logistic_pred)
pre=precision_score(y_test,logistic_pred,average="weighted",zero_division=0)
rec=recall_score(y_test,logistic_pred,average="weighted",zero_division=0)
f1=f1_score(y_test,logistic_pred,average="weighted",zero_division=0)
classification_results.append(["Logistic Regression",acc,pre,rec,f1])
print(classification_report(y_test,logistic_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,logistic_pred,cmap="Blues")
plt.title("Logistic Regression – Confusion Matrix"); plt.show()
print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")

## 3.4 Model 2 – Decision Tree Classifier\n**Step 1: define and train the model.**

In [ ]:
tree_model = Pipeline([
    ("preprocess",classification_preprocessor()),
    ("model",DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42, class_weight='balanced'))
])
tree_model.fit(X_train,y_train)
print("Decision Tree Classifier training completed.")

### Model 2 – Prediction

In [ ]:
tree_pred = tree_model.predict(X_test)
print("First 15 predictions:",tree_pred[:15])
print("First 15 actual labels:",y_test.iloc[:15].to_list())

### Model 2 – Evaluation and confusion matrix

In [ ]:
# Reuse the prediction already generated above for consistent evaluation.
acc=accuracy_score(y_test,tree_pred)
pre=precision_score(y_test,tree_pred,average="weighted",zero_division=0)
rec=recall_score(y_test,tree_pred,average="weighted",zero_division=0)
f1=f1_score(y_test,tree_pred,average="weighted",zero_division=0)
classification_results.append(["Decision Tree Classifier",acc,pre,rec,f1])
print(classification_report(y_test,tree_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,tree_pred,cmap="Blues")
plt.title("Decision Tree Classifier – Confusion Matrix"); plt.show()
print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")

## 3.5 Model 3 – KNN Classifier\n**Step 1: define and train the model.**

In [ ]:
knn_model = Pipeline([
    ("preprocess",classification_preprocessor()),
    ("model",KNeighborsClassifier(n_neighbors=min(11, len(X_train))))
])
knn_model.fit(X_train,y_train)
print("KNN Classifier training completed.")

### Model 3 – Prediction

In [ ]:
knn_pred = knn_model.predict(X_test)
print("First 15 predictions:",knn_pred[:15])
print("First 15 actual labels:",y_test.iloc[:15].to_list())

### Model 3 – Evaluation and confusion matrix

In [ ]:
# Reuse the prediction already generated above for consistent evaluation.
acc=accuracy_score(y_test,knn_pred)
pre=precision_score(y_test,knn_pred,average="weighted",zero_division=0)
rec=recall_score(y_test,knn_pred,average="weighted",zero_division=0)
f1=f1_score(y_test,knn_pred,average="weighted",zero_division=0)
classification_results.append(["KNN Classifier",acc,pre,rec,f1])
print(classification_report(y_test,knn_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,knn_pred,cmap="Blues")
plt.title("KNN Classifier – Confusion Matrix"); plt.show()
print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")

## 3.6 Model 4 – SVM Classifier\n**Step 1: define and train the model.**

In [ ]:
svm_model = Pipeline([
    ("preprocess",classification_preprocessor()),
    ("model",SVC(kernel='rbf', C=1.0, class_weight='balanced'))
])
svm_model.fit(X_train,y_train)
print("SVM Classifier training completed.")

### Model 4 – Prediction

In [ ]:
svm_pred = svm_model.predict(X_test)
print("First 15 predictions:",svm_pred[:15])
print("First 15 actual labels:",y_test.iloc[:15].to_list())

### Model 4 – Evaluation and confusion matrix

In [ ]:
# Reuse the prediction already generated above for consistent evaluation.
acc=accuracy_score(y_test,svm_pred)
pre=precision_score(y_test,svm_pred,average="weighted",zero_division=0)
rec=recall_score(y_test,svm_pred,average="weighted",zero_division=0)
f1=f1_score(y_test,svm_pred,average="weighted",zero_division=0)
classification_results.append(["SVM Classifier",acc,pre,rec,f1])
print(classification_report(y_test,svm_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,svm_pred,cmap="Blues")
plt.title("SVM Classifier – Confusion Matrix"); plt.show()
print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")

## 3.7 Model 5 – Naive Bayes\n**Step 1: define and train the model.**

In [ ]:
naive_bayes_model = Pipeline([
    ("preprocess",classification_preprocessor()),
    ("model",MultinomialNB())
])
naive_bayes_model.fit(X_train,y_train)
print("Naive Bayes training completed.")

### Model 5 – Prediction

In [ ]:
naive_bayes_pred = naive_bayes_model.predict(X_test)
print("First 15 predictions:",naive_bayes_pred[:15])
print("First 15 actual labels:",y_test.iloc[:15].to_list())

### Model 5 – Evaluation and confusion matrix

In [ ]:
# Reuse the prediction already generated above for consistent evaluation.
acc=accuracy_score(y_test,naive_bayes_pred)
pre=precision_score(y_test,naive_bayes_pred,average="weighted",zero_division=0)
rec=recall_score(y_test,naive_bayes_pred,average="weighted",zero_division=0)
f1=f1_score(y_test,naive_bayes_pred,average="weighted",zero_division=0)
classification_results.append(["Naive Bayes",acc,pre,rec,f1])
print(classification_report(y_test,naive_bayes_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,naive_bayes_pred,cmap="Blues")
plt.title("Naive Bayes – Confusion Matrix"); plt.show()
print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")

## 3.8 Model 6 – Extra – Random Forest Classifier\n**Step 1: define and train the model.**

In [ ]:
random_forest_model = Pipeline([
    ("preprocess",classification_preprocessor()),
    ("model",RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=42))
])
random_forest_model.fit(X_train,y_train)
print("Extra – Random Forest Classifier training completed.")

### Model 6 – Prediction

In [ ]:
random_forest_pred = random_forest_model.predict(X_test)
print("First 15 predictions:",random_forest_pred[:15])
print("First 15 actual labels:",y_test.iloc[:15].to_list())

### Model 6 – Evaluation and confusion matrix

In [ ]:
# Reuse the prediction already generated above for consistent evaluation.
acc=accuracy_score(y_test,random_forest_pred)
pre=precision_score(y_test,random_forest_pred,average="weighted",zero_division=0)
rec=recall_score(y_test,random_forest_pred,average="weighted",zero_division=0)
f1=f1_score(y_test,random_forest_pred,average="weighted",zero_division=0)
classification_results.append(["Extra – Random Forest Classifier",acc,pre,rec,f1])
print(classification_report(y_test,random_forest_pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,random_forest_pred,cmap="Blues")
plt.title("Extra – Random Forest Classifier – Confusion Matrix"); plt.show()
print(f"Accuracy={acc:.3f} | Precision={pre:.3f} | Recall={rec:.3f} | F1={f1:.3f}")

## 3.9 Classification model comparison

Compare all six original models against the majority-class baseline. Scores arise from the held-out cases and are not artificially chosen. **All case labels are synthetic administrative priorities, not actual bail classifications.**

In [ ]:
classification_comparison=pd.DataFrame(classification_results,
 columns=["Model","Accuracy","Precision","Recall","F1"])
display(classification_comparison.sort_values("F1",ascending=False))
classification_comparison.set_index("Model").plot(kind="bar",figsize=(12,5),ylim=(0,1))
plt.title("Classification model metric comparison"); plt.ylabel("Score"); plt.xticks(rotation=35,ha="right")
plt.tight_layout(); plt.show()
from sklearn.dummy import DummyClassifier
base_pred=baseline.predict(X_test)
print('Baseline accuracy:',round(accuracy_score(y_test,base_pred),3))
print('Highest model accuracy:',round(classification_comparison.Accuracy.max(),3))
if classification_comparison.Accuracy.min() >= .999:
    print('CHECK: all near-perfect results; audit target generation, splits and features.')
# Audit for exact case-ID and Act+Section lookup shortcuts; neither belongs in X.
assert X_train.index.intersection(X_test.index).empty
assert not set(['Act','Section','Case_ID']).intersection(X.columns)
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
cv_model=Pipeline([('preprocess',classification_preprocessor()),('model',LogisticRegression(max_iter=1000))])
cv_scores=cross_val_score(cv_model,X,y,cv=cv,scoring='accuracy',n_jobs=1)
print('3-fold stratified CV (Logistic Regression):',np.round(cv_scores,3),'mean=',round(cv_scores.mean(),3))


# Unit 3 – Regression: one-month population forecasting
Target is next month's facility population. We split chronologically so later months are held out. All features used below are available at the feature month; the target is next month.

## 3.10 Regression split and preprocessing

In [ ]:
reg=reg.dropna(subset=["Target_Next_Month_Population"]).sort_values("Feature_Month").copy()
num_reg=["Capacity","Population_Current","Population_Lag_1M","Population_Lag_3M",
         "Population_Lag_6M","Admissions_Current_Month","Releases_Current_Month",
         "Transfers_In_Current_Month","Transfers_Out_Current_Month","Calendar_Month"]
cat_reg=["State_UT","Facility_Type"]
unique_months=np.sort(reg.Feature_Month.unique())
split_month=unique_months[int(len(unique_months)*.8)]
train_r=reg[reg.Feature_Month<split_month]; test_r=reg[reg.Feature_Month>=split_month]
Xtr=train_r[num_reg+cat_reg]; Xte=test_r[num_reg+cat_reg]
yr=train_r.Target_Next_Month_Population; yt=test_r.Target_Next_Month_Population
def reg_preprocessor(numeric=num_reg):
    return ColumnTransformer([
      ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),
                       ("scale",StandardScaler())]),numeric),
      ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                       ("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_reg)])
print("Training period:",train_r.Feature_Month.min(),"to",train_r.Feature_Month.max())
print("Testing period:",test_r.Feature_Month.min(),"to",test_r.Feature_Month.max())

## 3.11 Regression evaluation helper

In [ ]:
regression_results=[]
def regression_evaluation(name,actual,predicted):
    mae=mean_absolute_error(actual,predicted)
    mse=mean_squared_error(actual,predicted)
    rmse=np.sqrt(mse); r2=r2_score(actual,predicted)
    regression_results.append([name,mae,mse,rmse,r2])
    print(f"{name}: MAE={mae:.2f}, MSE={mse:.2f}, RMSE={rmse:.2f}, R2={r2:.3f}")
    plt.figure(figsize=(6,5)); plt.scatter(actual,predicted,alpha=.3)
    lo=min(actual.min(),np.min(predicted)); hi=max(actual.max(),np.max(predicted))
    plt.plot([lo,hi],[lo,hi],"r--"); plt.xlabel("Actual"); plt.ylabel("Predicted")
    plt.title(name+" – Actual vs predicted"); plt.show()
    residual=np.asarray(actual)-np.asarray(predicted)
    plt.figure(figsize=(6,4)); plt.scatter(predicted,residual,alpha=.3); plt.axhline(0,color="red",ls="--")
    plt.xlabel("Predicted"); plt.ylabel("Residual"); plt.title(name+" – Residual plot"); plt.show()

## 3.12 Model 1 – Simple Linear Regression
Uses one predictor (current population) to predict next-month population.

In [ ]:
simple_lr=Pipeline([("prep",ColumnTransformer([("pop",Pipeline([
    ("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),
    ["Population_Current"])])),("model",LinearRegression())])
simple_lr.fit(Xtr[["Population_Current"]],yr)
print("Simple Linear Regression trained.")

### Simple Linear Regression – prediction

In [ ]:
simple_pred=simple_lr.predict(Xte[["Population_Current"]])
display(pd.DataFrame({"Actual":yt.iloc[:10].values,"Predicted":simple_pred[:10]}))

### Simple Linear Regression – evaluation and graphs

In [ ]:
regression_evaluation("Simple Linear Regression",yt,simple_pred)

## 3.13 Model 2 – Multiple Linear Regression\n**Step 1: define and train.**

In [ ]:
multiple_lr=Pipeline([("prep",reg_preprocessor()),("model",LinearRegression())])
multiple_lr.fit(Xtr,yr)
print("Multiple Linear Regression trained.")

### Model 2 – Prediction

In [ ]:
multiple_lr_pred=multiple_lr.predict(Xte)
display(pd.DataFrame({"Actual":yt.iloc[:10].values,"Predicted":multiple_lr_pred[:10]}))

### Model 2 – Evaluation and graphs

In [ ]:
regression_evaluation("Multiple Linear Regression",yt,multiple_lr_pred)

## 3.14 Model 3 – Decision Tree Regression\n**Step 1: define and train.**

In [ ]:
decision_tree_reg=Pipeline([("prep",reg_preprocessor()),("model",DecisionTreeRegressor(max_depth=8,min_samples_leaf=8,random_state=42))])
decision_tree_reg.fit(Xtr,yr)
print("Decision Tree Regression trained.")

### Model 3 – Prediction

In [ ]:
decision_tree_reg_pred=decision_tree_reg.predict(Xte)
display(pd.DataFrame({"Actual":yt.iloc[:10].values,"Predicted":decision_tree_reg_pred[:10]}))

### Model 3 – Evaluation and graphs

In [ ]:
regression_evaluation("Decision Tree Regression",yt,decision_tree_reg_pred)

## 3.15 Model 4 – Support Vector Regression\n**Step 1: define and train.**

In [ ]:
svr_reg=Pipeline([("prep",reg_preprocessor()),("model",SVR(kernel='rbf',C=10,epsilon=5))])
svr_reg.fit(Xtr,yr)
print("Support Vector Regression trained.")

### Model 4 – Prediction

In [ ]:
svr_reg_pred=svr_reg.predict(Xte)
display(pd.DataFrame({"Actual":yt.iloc[:10].values,"Predicted":svr_reg_pred[:10]}))

### Model 4 – Evaluation and graphs

In [ ]:
regression_evaluation("Support Vector Regression",yt,svr_reg_pred)

## 3.16 Model 5 – Extra – Random Forest Regression\n**Step 1: define and train.**

In [ ]:
random_forest_reg=Pipeline([("prep",reg_preprocessor()),("model",RandomForestRegressor(n_estimators=150,max_depth=10,min_samples_leaf=5,random_state=42))])
random_forest_reg.fit(Xtr,yr)
print("Extra – Random Forest Regression trained.")

### Model 5 – Prediction

In [ ]:
random_forest_reg_pred=random_forest_reg.predict(Xte)
display(pd.DataFrame({"Actual":yt.iloc[:10].values,"Predicted":random_forest_reg_pred[:10]}))

### Model 5 – Evaluation and graphs

In [ ]:
regression_evaluation("Extra – Random Forest Regression",yt,random_forest_reg_pred)

## 3.17 Regression model comparison

In [ ]:
regression_comparison=pd.DataFrame(regression_results,columns=["Model","MAE","MSE","RMSE","R2"])
display(regression_comparison.sort_values("RMSE"))
regression_comparison.set_index("Model")[["MAE","RMSE"]].plot(kind="bar",figsize=(11,5))
plt.title("Regression comparison – lower error is better"); plt.ylabel("Error")
plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

## 3.18 One-month predicted occupancy and capacity warning (aggregate planning demo)

In [ ]:
# Choose the model with lowest RMSE from the held-out time period.
best_name=regression_comparison.sort_values("RMSE").iloc[0].Model
pred_map={"Simple Linear Regression":simple_pred,
          "Multiple Linear Regression":multiple_lr.predict(Xte),
          "Decision Tree Regression":decision_tree_reg.predict(Xte),
          "Support Vector Regression":svr_reg.predict(Xte),
          "Extra – Random Forest Regression":random_forest_reg.predict(Xte)}
chosen=np.maximum(0,pred_map[best_name])
forecast=test_r[["Facility_ID","State_UT","Facility_Type","Feature_Month","Capacity"]].copy()
forecast["Predicted_Next_Month_Population"]=np.rint(chosen).astype(int)
forecast["Predicted_Occupancy_Pct"]=forecast.Predicted_Next_Month_Population/forecast.Capacity*100
forecast["Capacity_Flag"]=np.where(forecast.Predicted_Occupancy_Pct>100,"Above capacity","At/below capacity")
print("Selected by lowest test RMSE:",best_name)
display(forecast.head(15))

# Unit 5 – Unsupervised Learning

## 5.1 Facility-level feature aggregation for clustering

In [ ]:
facility_features=monthly_clean.groupby(["Facility_ID","State_UT","Facility_Type"],as_index=False).agg(
    Mean_Population=("Population","mean"),Mean_Occupancy=("Occupancy_Recalculated","mean"),
    Max_Occupancy=("Occupancy_Recalculated","max"),Mean_Admissions=("Admissions","mean"),
    Mean_Releases=("Releases","mean"),Capacity=("Capacity","first"))
cluster_cols=["Mean_Population","Mean_Occupancy","Max_Occupancy","Mean_Admissions","Mean_Releases","Capacity"]
Xcluster=StandardScaler().fit_transform(facility_features[cluster_cols])
display(facility_features.head())

## 5.2 K-Means clustering – fit, predict labels, evaluate and plot

In [ ]:
# A silhouette score requires 2 through (n_samples-1) nonempty clusters.
if len(Xcluster) < 3:
    raise ValueError("K-Means requires at least 3 facility rows for this demonstration")
kmeans=KMeans(n_clusters=min(3,len(Xcluster)-1),random_state=42,n_init=10)
klabels=kmeans.fit_predict(Xcluster)
facility_features["KMeans_Cluster"]=klabels
k_count=len(np.unique(klabels))
ksil=silhouette_score(Xcluster,klabels) if 2<=k_count<len(Xcluster) else np.nan
print("K-Means silhouette score:",round(ksil,3) if np.isfinite(ksil) else "Not defined")
display(facility_features.groupby("KMeans_Cluster")[cluster_cols].mean().round(2))
plt.figure(figsize=(7,5)); plt.scatter(facility_features.Mean_Occupancy,facility_features.Mean_Population,c=klabels)
plt.xlabel("Mean occupancy (%)"); plt.ylabel("Mean population"); plt.title("K-Means facility clusters"); plt.show()

## 5.3 Hierarchical clustering – fit, evaluate and plot

In [ ]:
if len(Xcluster) < 3:
    raise ValueError("Hierarchical clustering requires at least 3 facility rows")
hier=AgglomerativeClustering(n_clusters=min(3,len(Xcluster)-1),linkage="ward")
hlabels=hier.fit_predict(Xcluster)
h_count=len(np.unique(hlabels))
hsil=silhouette_score(Xcluster,hlabels) if 2<=h_count<len(Xcluster) else np.nan
print("Hierarchical silhouette score:",round(hsil,3) if np.isfinite(hsil) else "Not defined")
plt.figure(figsize=(7,5)); plt.scatter(facility_features.Mean_Occupancy,facility_features.Mean_Population,c=hlabels)
plt.xlabel("Mean occupancy (%)"); plt.ylabel("Mean population"); plt.title("Hierarchical clusters"); plt.show()

## 5.4 DBSCAN – fit, identify noise and evaluate when valid

In [ ]:
db=DBSCAN(eps=1.5,min_samples=5)
dlabels=db.fit_predict(Xcluster)
facility_features["DBSCAN_Cluster"]=dlabels
valid=dlabels!=-1
if 2 <= len(set(dlabels[valid])) < int(valid.sum()):
    print("DBSCAN silhouette (noise excluded):",round(silhouette_score(Xcluster[valid],dlabels[valid]),3))
else:
    print("Silhouette not defined: DBSCAN found fewer than two non-noise clusters.")
print("Noise points:",int((dlabels==-1).sum()))
plt.figure(figsize=(7,5)); plt.scatter(facility_features.Mean_Occupancy,facility_features.Mean_Population,c=dlabels)
plt.xlabel("Mean occupancy (%)"); plt.ylabel("Mean population"); plt.title("DBSCAN clusters (-1 = noise)"); plt.show()

## 5.5 PCA – two-dimensional cluster visualization

In [ ]:
pca2=PCA(n_components=2,random_state=42)
xy=pca2.fit_transform(Xcluster)
print("PCA variance explained:",pca2.explained_variance_ratio_)
plt.figure(figsize=(7,5)); plt.scatter(xy[:,0],xy[:,1],c=klabels)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA view of K-Means clusters"); plt.show()

## 5.6 t-SNE overview and visualization

In [ ]:
# t-SNE requires perplexity strictly smaller than the number of rows.
# For fewer than 3 rows, a 2D embedding is not informative.
if len(Xcluster) < 3:
    print("t-SNE skipped: need at least 3 facility samples for a 2D view")
else:
    perp=min(30, max(1, (len(Xcluster)-1)//3), len(Xcluster)-1)
    ts=TSNE(n_components=2,random_state=42,perplexity=perp,init="pca",learning_rate="auto")
    txy=ts.fit_transform(Xcluster)
    plt.figure(figsize=(7,5)); plt.scatter(txy[:,0],txy[:,1],c=klabels)
    plt.title("t-SNE visualization"); plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2"); plt.show()

## 5.7 Association rule mining – Apriori

In [ ]:
# Quantile bin edges may collapse when values are repeated; make label
# counts match the number of resulting intervals.
def robust_quantile_bands(values, labels):
    values=pd.to_numeric(values,errors="coerce")
    if values.notna().sum()==0:
        return pd.Series("Unknown",index=values.index,dtype="object")
    _,edges=pd.qcut(values,q=len(labels),retbins=True,duplicates="drop")
    n_bins=len(edges)-1
    if n_bins == 0:
        return pd.Series("Medium",index=values.index,dtype="object")
    if n_bins == len(labels):
        chosen=labels
    elif n_bins == 1:
        chosen=[labels[len(labels)//2]]
    else:
        chosen=[labels[0],labels[-1]] if n_bins==2 else labels[:n_bins]
    return pd.cut(values,bins=edges,labels=chosen,include_lowest=True).astype("object").fillna("Unknown")

basket_source=facility_features[["Mean_Occupancy","Capacity","Mean_Admissions"]].copy()
basket_source["Occupancy_Band"]=pd.cut(basket_source.Mean_Occupancy,
    [-np.inf,70,90,100,np.inf],labels=["Low","Moderate","High","Over_Capacity"])
basket_source["Capacity_Band"]=robust_quantile_bands(basket_source.Capacity,["Small","Medium","Large"])
basket_source["Admissions_Band"]=robust_quantile_bands(basket_source.Mean_Admissions,["Low","Medium","High"])
basket=pd.get_dummies(basket_source[["Occupancy_Band","Capacity_Band","Admissions_Band"]].astype(str)).astype(bool)
itemsets=apriori(basket,min_support=.08,use_colnames=True)
if len(itemsets):
    rules=association_rules(itemsets,metric="confidence",min_threshold=.5)
    if len(rules):
        display(rules.sort_values(["lift","confidence"],ascending=False).head(15))
    else:
        print("Frequent itemsets found, but none satisfy the confidence threshold.")
else:
    print("No frequent itemsets at this support; lower min_support slightly.")

# Unit 3 – Model concepts, validation and final interpretation

## Loss function, overfitting, underfitting, bias and variance

- **Loss/cost function:** quantifies prediction error. Regression commonly uses squared error; classification often uses log loss.
- **Overfitting:** excellent training fit but poor generalization to unseen data.
- **Underfitting:** model too simple to capture patterns.
- **Bias:** error from overly restrictive assumptions.
- **Variance:** sensitivity to training-data changes.
- **Cross-validation:** repeated train/validation splits estimate generalization.
- **Bootstrap:** sampling with replacement to estimate variability.

## Cross-validation demonstration (classification training data only)

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold
# Stratified K-fold needs at least one example of every label in each fold.
cv_splits=min(5,len(X_train))
if cv_splits < 2:
    print("Cross-validation skipped: fewer than 2 training observations")
else:
    min_class_count=y_train.value_counts().min()
    if min_class_count >= 2:
        cv=StratifiedKFold(n_splits=min(cv_splits,int(min_class_count)),shuffle=True,random_state=42)
    else:
        cv=KFold(n_splits=cv_splits,shuffle=True,random_state=42)
        print("Rare classes: using regular K-Fold instead of stratified K-Fold")
    cv_pipe=Pipeline([("prep",classification_preprocessor()),
                      ("model",LogisticRegression(max_iter=1500,class_weight="balanced"))])
    scores=cross_val_score(cv_pipe,X_train,y_train,cv=cv,scoring="f1_weighted")
    print("Fold F1 scores:",np.round(scores,3))
    print("Mean CV F1:",round(scores.mean(),3),"Std:",round(scores.std(),3))

## Bootstrap resampling – confidence interval for a mean (demonstration)

In [ ]:
# Bootstrap the mean monthly population; this illustrates resampling with replacement.
vals=monthly_clean.Population.dropna().to_numpy()
boot_rng=np.random.default_rng(42)
boot_means=[boot_rng.choice(vals,size=len(vals),replace=True).mean() for _ in range(500)]
print("Observed mean:",round(vals.mean(),2))
print("Bootstrap 95% interval:",np.round(np.percentile(boot_means,[2.5,97.5]),2))
plt.figure(figsize=(7,4)); plt.hist(boot_means,bins=30); plt.title("Bootstrap distribution of mean population")
plt.xlabel("Resampled mean"); plt.show()

## Final project conclusions and limitations

This notebook retains the original Units 1, 2, 3 and 5, all six classification models, regression models, clustering, PCA, t-SNE, and Apriori. Classification now predicts **synthetic administrative case-review priority**, not bailability. The simulated label contains intentionally noisy associations with case-level process data, so scores should be reported as observed—not adjusted to look realistic. Data are entirely fictional and **must never be used to decide bail, detention, sentencing or legal eligibility**. Time-series forecasting and clustering are also demonstrations and require real validation before deployment.

1. Unit 1 concepts and data types are introduced.
2. Unit 2 performs EDA, cleaning, smoothing, encoding, scaling, feature engineering and PCA.
3. Unit 3 trains classification and regression models in separate, ordered cells and evaluates each independently.
4. Unit 5 applies K-Means, hierarchical clustering, DBSCAN, silhouette score, PCA, t-SNE and association rule mining.
5. Unit 4 neural networks are intentionally omitted.
6. The workbook is synthetic. Model metrics are not real-world performance claims.
7. Bailability must be determined from the exact offence and current official law. The ML demonstration must never be used to decide an individual's bail or detention.
8. Facility forecasting and overcrowding flags are aggregate planning demonstrations. Prison assignment and officer selection require verified operational data and authorized human decision-making.